# Factor Template

This is a factor working pipeline template from reading, calculation to evaluation.

## Factor Design

You are able to construction a factor by natural language, but you need to state it in a very specific way. The following block is a example. And then, you are able to copy  the definition text string to generate the factor computing code.

**NOTE:** This notebook is only available for factors that can be calculated in matrix. So if a factor needs complicated calculation or different procession for different stock, DON'T USE THIS BLOCK TO GENERATE FACTOR. Instead, you need to turn to `generate.py` for help.

In [ ]:
import os
import dotenv
from pathlib import Path

import pandas as pd
import numpy as np

from parquool import Agent
from factool import DuckPQSource

dotenv.load_dotenv()

## XX Factor

**Definition**：

Brief definition for the factor

**Steps**：

1. Specifications for data extraction
2. Provide equations or algorithms to produce the factor
3. Filters or exceptions for setting factor to NaN

In [ ]:
agent = Agent(
    instructions=Path("../docs/CODE_GENERATOR.md").read_text(encoding="utf-8")
)
prompt = r""""""  # Copy your full definition text here
result = agent.run(prompt)

Then, you just copy generated code to a new code cell, and run it!

In [ ]:
df = ...

## Optional: Factor Saving

After generating factor, it makes things easier if you save the factor data to disk. Feel free to use `DuckParquetSource` to save any standarized data. `DuckParquetSource.save` helps you with the affairs in saving factor data. There are two available standarized data form: 1. Wide table with different stock codes for each column, and different time lables for each row; 2. Long table with two level index, the first level should be time index, and the second level should be code index. Once you prepare data in the above form, you can pass the data to `DuckParquetSource.save`. Bear in mind that if you use form 1, you need to pass another parameter called `name` to specify the name of the factor, or default name `factor` will be applied.

The `processors` parameter provide a data cleaning way before saving. If set to `None`, `zscore` and `madoutlier` with `dev=5` will be applied

In [ ]:
from pathlib import Path


table_name = ""  # Apply your factor table name here
processors = None
DuckPQSource(Path(os.getenv("FACTOR_DATA_PATH"))).save(
    table_name, df, processors=processors
)

## Factor Evaluation

`Evaluator` is an important component in `factool`. You can initialize it with simply one-line-code. Then you can apply multiple methods to evaluate the factor with different parameters. Note that factor here can be multiple, you can set different factor in a list like `[df1, df2, ...]`

In [ ]:
import os
import dotenv
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display_markdown, Markdown

import quool
from factool import DuckPQSource, Evaluator

dotenv.load_dotenv()

### Parameter Setting

- factor_name: name used for connecting to database
- begin: evaluation begin time
- end: evaluation end time
- ptype: price type for calculating return
- horizon: future return period
- skip_horizon: whether to skip non-rebalancing day
- ic_method: way to perform ic computation, choices between "spearman" and "pearson"
- n_groups: how many groups to divide
- bucketing_mode: way to perform grouping, choices between "single", "independent" and "conditional"
- ts_window: time series rolling regression window
- ts_min_obs: least observations in rolling window
- ts_n_jobs: how many parallel cpu used when rolling in time series
- cs_add_intercept: whether to add interception when cross section regression
- cs_cov_type: covariance type calculated when cross section regression, between "white" and "none"
- cs_white_type: white estimator, choice between "HC0" and "HC1"
- feasible & weight: feasible return mask and portfolio weight

In [ ]:
table_name = r""  # Apply your factor name here
begin = "2015-01-01"
end = "2025-06-30"
ptype = "close_post"
horizon = 5
skip_horizon = True
ic_method = "spearman"
n_groups = 10
bucketing_mode = "single"
ts_window = 252
ts_min_obs = 60
ts_intercept = True
ts_n_jobs = 1
cs_add_intercept = True
cs_cov_type = "white"
cs_white_type = "HC1"
results = {}

# Data preparation
ds = DuckPQSource(Path(os.getenv("DATASET_PATH")))
ds.register("quotes_day")
ds.register("instruments_info")
ds_data = ds.query(f"""
SELECT
    q.date AS date,
    q.code AS code,
    q.{ptype} AS price,
    (
        q.high > q.limit_down
        AND q.low  < q.limit_up
        AND COALESCE(q.st, false) = false
        AND COALESCE(q.suspended, false) = false
        AND datediff('day', i.listed_date, q.date) > 90 -- Exclude newly listed stock
    ) AS tradable_mask
FROM quotes_day AS q
JOIN instruments_info AS i
    ON q.code = i.code
WHERE q.date >= '{begin}' AND q.date <= '{end}'
""")
feasible = ds_data.pivot(index="date", columns="code", values="tradable_mask")
price = ds_data.pivot(index="date", columns="code", values="price")
weight = None
factor_source = DuckPQSource(Path(os.getenv("FACTOR_DATA_PATH")))

### Factor Name Setting

In [ ]:
name = ""  # Apply factor name here
df = factor_source.get_factor(table_name, name, begin=begin, end=end)
e = Evaluator(factor=df, price=price)
results[tuple(e._factors.keys())] = result = {}

### Coverage Evaluation

First of all, we should explore the integrity of the factor data. Data coverage define the percentage of non-NA data to the full dataset. If the coverage is too low, meaning available data is insufficient and the following tests may be biased. 

In [ ]:
cov_res = {}
for name, df in e._factors.items():
    coverage = df.count(axis=1) / price.count(axis=1)
    coverage.plot(title=f"Coverage of {name}", figsize=(20, 10))
    cov_res[name] = coverage.mean()
result["coverage"] = pd.Series(cov_res)

### IC Evaluation

IC shows information coeffiency of factor, which means the spearman or pearson relation coeffiency between factor and future return. When the IC value is positive, the larger factor it is, the higher future return it expects. And we also refer to the t-value of IC, which shows us the stability of the relationship between factor and future return

In [ ]:
e.get_info_coef(horizon=horizon, skip_horizon=skip_horizon, method=ic_method)
result["ic"] = pd.concat(
    [e.ic.mean(), e.ic.mean() / (e.ic.std() / np.sqrt(e.ic.shape[0]))],
    axis=1,
    keys=["ic_mean", "ic_tval"],
)

In [ ]:
pd.concat(
    [
        e.ic,
        e.ic.rolling(20).mean().add_suffix(" rolling 20 mean"),
        e.ic.cumsum().add_suffix(" cumsum"),
    ],
    axis=1,
).plot(
    figsize=(20, 10),
    secondary_y=(e.ic.columns + " cumsum").to_list(),
    title="IC & IC Rolling Mean & IC Cumsum",
)
e.ic.resample("YE").mean().plot.bar(figsize=(20, 10), title="Year IC Average")
e.ic.resample("ME").mean().plot.bar(figsize=(35, 10), title="Month IC Average")
display(result["ic"].to_frame("IC-Mean"))

### Grouping Evaluation

Grouping is a classic way to compute the factor return. By ordering stocks with factor value,dividing stocks into n groups, we can contruct a portfolio longing on the largest group on the factor and shorting on the smallest group on the factor. So the return of the portfolio will be $ R(large) - R(small) $. Within the group, we can simpling setting the weights to equal or using market value. And in this part, we can see the performance of each portfolio group. We will focus on the monotonicity of the returns on each group. And on the time series point, we can see the average return and the t-value of the return series.

In [ ]:
e.get_group_returns(
    n=n_groups,
    horizon=horizon,
    skip_horizon=skip_horizon,
    mode=bucketing_mode,
    feasible=feasible,
    weight=weight,
)
factor_return = e.sorted_factor_return
group_returns = pd.concat(
    [gr.groupby(level=0).mean() for gr in e.group_returns.values()] + [factor_return],
    axis=1,
)
group_returns_mean = group_returns.mean()
group_returns_t = group_returns_mean / (
    group_returns.std() / np.sqrt(group_returns.shape[0])
)
result["group_return"] = pd.concat(
    [group_returns_mean, group_returns_t],
    axis=1,
    keys=["group_return_mean", "group_return_t"],
)

In [ ]:
for name in e._factors.keys():
    group_returns_mean[[f"{name}({i + 1})" for i in range(n_groups)]].plot.bar(
        figsize=(20, 10), title=f"Return for {name}"
    )
group_value = (1 + group_returns.shift(1).dropna(how="all", axis=0).fillna(0)).cumprod()
for name in e._factors.keys():
    group_value[[f"{name}({i + 1})" for i in range(n_groups)]].plot(
        figsize=(20, 10), title=f"Group Cumulative Value {name}"
    )
group_eval = group_value.apply(quool.Evaluator.evaluate)
display(group_eval)
display(group_returns_t.to_frame("Group Return T Value"))

### Time Series Regression

After grouping, factor returns can be estimated by using $ R(large) - R(small) $. Tracing back to CAPM model, we can construct a time series regression using some time window: $ E[R_{t}] = \alpha + \beta \cdot f_t $. where $ R_{i, t} $ is the expected return on time t for stock, $ f_t $ is the factor return on time t. So for each stock, we can get a regression beta standing for the factor exposure. For rolling window, we can get multiple factor exposure. Apply average on that time series value, we focus on the relationship between averaged factor exposure and averaged future reutrn; coeffiency between this two time series value; and t-value for factor exposure to see the stability.

In [ ]:
e.get_factor_exposure(
    horizon=horizon,
    feasible=feasible,
    window=ts_window,
    min_obs=ts_min_obs,
    intercept=ts_intercept,
    n_jobs=ts_n_jobs,
)
factor_exposure_mean = e.factor_exposure.groupby(level=0).mean()
factor_exposure_mean_t = e.factor_exposure_t.groupby(level=0).mean()
concated = pd.concat(
    [
        factor_exposure_mean,
        e._future_return(horizon).mean().to_frame(f"{horizon}d return"),
    ],
    axis=1,
)

In [ ]:
f_stats = {}
for nm in e._factors.keys():
    tsic = (
        e.factor_exposure[nm]
        .unstack(level=0)
        .corrwith(e._future_return(horizon=horizon, skip=False), axis=1)
        .dropna()
    )
    pd.concat(
        [
            tsic.to_frame(nm),
            tsic.rolling(20).mean().to_frame(nm).add_suffix(" rolling 20 mean"),
            tsic.cumsum().to_frame(nm).add_suffix(" cumsum"),
        ],
        axis=1,
    ).plot(
        figsize=(20, 10),
        title="Factor Exposure Correlation with Future Return",
        secondary_y=[f"{nm} cumsum"],
    )
    f_stats[f"{nm} TSIC-Mean"] = tsic.mean()
    f_stats[f"{nm} TSIC-TValue"] = tsic.mean() / (tsic.std() / np.sqrt(tsic.shape[0]))
display(pd.Series(f_stats).to_frame("Factor Exposure TSIC Stats"))
pd.plotting.scatter_matrix(
    concated.iloc[:, 1:],
    figsize=(20, 5 * concated.shape[1] - 1),
    hist_kwds={"bins": 100},
)
display(concated)

### Cross Section Regression

Another way of solving factor is just using factor value as factor exposure. Appling cross section regression on each time point is a classic way called Fama-Macbeth regression. By the residual of each regression, we can testify whether the factor can explain returns for each stock well. By the regression coefficiency, we can see how the factor return varies with time.

In [ ]:
e.cross_sectional_regression(
    horizon=horizon,
    feasible=feasible,
    weight=weight,
    add_intercept=cs_add_intercept,
    cov_type=cs_cov_type,
    white_type=cs_white_type,
)
concated = pd.concat(
    [
        e.factor_premia.iloc[:, 1:],
        e.factor_premia.iloc[:, 1:].cumsum().add_suffix("-Cumsum"),
    ]
)


In [ ]:
concated.plot(
    figsize=(20, 10),
    secondary_y=(e.factor_premia.columns + "-Cumsum").to_list(),
    title="Factor Return for Each Day & Cumulative Return",
)
display(pd.concat([e.factor_premia, e.factor_r2, e.factor_premia_t], axis=1))

### Overall Test Results

In [ ]:
for key, val in results.items():
    display_markdown(Markdown(f"## {key}"))
    display(val["coverage"].to_frame("coverage"))
    display(val["ic"])
    display(val["group_return"])